In [1]:
# ==============================================================================
# 1. DATA CLEANING AND MERGING SCRIPT
#
# Reads raw data from ../01_raw_data/, performs the following steps:
#   - Combines 'date' and 'time' into a single datetime index.
#   - Handles duplicate timestamps by aggregating data.
#   - Renames sensor-specific columns with a prefix (e.g., 'temp' -> 'fridge_temp').
#   - Merges all 7 datasets into a single DataFrame.
#   - Consolidates the 'label' and 'type' columns.
#   - Fills any missing values.
# Saves the final, clean DataFrame to ../03_cleaned_data/
# ==============================================================================

import pandas as pd
import os
import glob
import numpy as np

# --- Configuration ---
# Note the relative paths, which are correct for our project structure.
raw_data_directory = '../01_raw_data/Train_Test_IoT_dataset/'
output_directory = '../03_cleaned_data/'
output_filename = 'Cleaned_IoT_Dataset.csv'
output_path = os.path.join(output_directory, output_filename)

print(f"Searching for raw CSV files in: {raw_data_directory}")

try:
    csv_files = glob.glob(os.path.join(raw_data_directory, "*.csv"))
    
    if not csv_files:
        raise FileNotFoundError(f"No CSV files were found in the specified directory. Please check the path.")

    print("\n--- Found the following CSV files to merge ---")
    for f in csv_files:
        print(os.path.basename(f))

    all_dfs = []
    for f in csv_files:
        device_name = os.path.basename(f).replace('Train_Test_IoT_', '').replace('.csv', '').lower()
        temp_df = pd.read_csv(f)
        
        # Step 1: Create a proper datetime column and handle errors
        temp_df['datetime'] = pd.to_datetime(temp_df['date'] + ' ' + temp_df['time'], errors='coerce')
        temp_df = temp_df.dropna(subset=['datetime']) # Drop rows where date/time was invalid
        temp_df = temp_df.drop(columns=['date', 'time'])
        
        # Step 2: Handle duplicate timestamps by aggregating
        agg_dict = {col: 'mean' for col in temp_df.select_dtypes(include=np.number).columns}
        if 'label' in agg_dict:
            agg_dict['label'] = 'max' # Prioritize 'attack' label
        for col in temp_df.select_dtypes(include='object').columns:
            agg_dict[col] = 'last' # Take last entry for text columns
        temp_df = temp_df.groupby('datetime').agg(agg_dict)
        
        # Step 3: Rename all columns with a device-specific prefix
        temp_df = temp_df.add_prefix(f"{device_name}_")
        
        all_dfs.append(temp_df)
    
    print(f"\nSuccessfully loaded and aggregated {len(all_dfs)} device datasets.")

    # Step 4: Merge all DataFrames on the unique datetime index
    df_merged = pd.concat(all_dfs, axis=1, join='outer')

    # Step 5: Consolidate 'label' and 'type' columns from all devices
    label_cols = [col for col in df_merged.columns if 'label' in col]
    type_cols = [col for col in df_merged.columns if 'type' in col]

    df_merged['label'] = df_merged[label_cols].max(axis=1)
    
    df_types = df_merged[type_cols].replace('normal', pd.NA)
    # Use backfill then forward fill to get the first available attack type
    df_merged['type'] = df_types.bfill(axis=1).ffill(axis=1).iloc[:, 0].fillna('normal')
    
    # Drop the now-redundant prefixed label and type columns
    df_merged = df_merged.drop(columns=label_cols + type_cols)

    # Step 6: Handle missing sensor data from the merge
    # We use forward fill then backward fill to ensure no gaps remain
    df_combined = df_merged.ffill().bfill()
    
    # Step 7: Save the clean dataset to the output directory
    os.makedirs(output_directory, exist_ok=True) # Ensure the output directory exists
    df_combined.to_csv(output_path)
    print(f"\n--- SUCCESS! ---")
    print(f"Cleaned and merged dataset saved to: {output_path}")

    # --- Final Inspection ---
    print("\n--- Final Dataset Info ---")
    print(f"Shape: {df_combined.shape[0]} rows, {df_combined.shape[1]} columns.")
    print("\nColumn Names:")
    print(df_combined.columns.tolist())
    print("\nLabel Distribution:")
    print(df_combined['label'].value_counts())
    print("\nNull Values Check:")
    print(df_combined.isnull().sum().sum())


except Exception as e:
    print(f"\nAn error occurred: {e}")

Searching for raw CSV files in: ../01_raw_data/Train_Test_IoT_dataset/

--- Found the following CSV files to merge ---
Train_Test_IoT_Fridge.csv
Train_Test_IoT_Garage_Door.csv
Train_Test_IoT_GPS_Tracker.csv
Train_Test_IoT_Modbus.csv
Train_Test_IoT_Motion_Light.csv
Train_Test_IoT_Thermostat.csv
Train_Test_IoT_Weather.csv


C:\Users\arceu\AppData\Local\Temp\ipykernel_21920\4219604218.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_df['datetime'] = pd.to_datetime(temp_df['date'] + ' ' + temp_df['time'], errors='coerce')
C:\Users\arceu\AppData\Local\Temp\ipykernel_21920\4219604218.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_df['datetime'] = pd.to_datetime(temp_df['date'] + ' ' + temp_df['time'], errors='coerce')
C:\Users\arceu\AppData\Local\Temp\ipykernel_21920\4219604218.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_df['datetime'] = pd.to_datetime(temp_df['da


Successfully loaded and aggregated 7 device datasets.


C:\Users\arceu\AppData\Local\Temp\ipykernel_21920\4219604218.py:74: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_merged['type'] = df_types.bfill(axis=1).ffill(axis=1).iloc[:, 0].fillna('normal')



--- SUCCESS! ---
Cleaned and merged dataset saved to: ../03_cleaned_data/Cleaned_IoT_Dataset.csv

--- Final Dataset Info ---
Shape: 73902 rows, 19 columns.

Column Names:
['fridge_fridge_temperature', 'fridge_temp_condition', 'garage_door_door_state', 'garage_door_sphone_signal', 'gps_tracker_latitude', 'gps_tracker_longitude', 'modbus_FC1_Read_Input_Register', 'modbus_FC2_Read_Discrete_Value', 'modbus_FC3_Read_Holding_Register', 'modbus_FC4_Read_Coil', 'motion_light_motion_status', 'motion_light_light_status', 'thermostat_current_temperature', 'thermostat_thermostat_status', 'weather_temperature', 'weather_pressure', 'weather_humidity', 'label', 'type']

Label Distribution:
label
1.0    63485
0.0    10417
Name: count, dtype: int64

Null Values Check:
0
